In [24]:
import pandas as pd
import numpy as np

from tqdm import tqdm; tqdm.pandas()
import sys

## Importing Data

In [2]:
df = pd.read_parquet('E:/GZ-DESI/data/raw-cats/_desi_pytorch_v5_hpv2_train_all_notest_ml_catalog_x5_advanced.parquet', engine = 'pyarrow', columns = ['id_str', 'hdf5_loc', 'merging_none_fraction', 'merging_minor-disturbance_fraction', 'merging_major-disturbance_fraction', 'merging_merger_fraction'])

In [3]:
df_red = df[['id_str', 'hdf5_loc','merging_none_fraction', 'merging_minor-disturbance_fraction', 'merging_major-disturbance_fraction', 'merging_merger_fraction']]

## Selecting Interacting Galaxies

In [4]:
df_ints = (
    df_red
    .query('merging_none_fraction <= 0.25')
    .drop(columns = ['merging_none_fraction'])
)

## Renormalizing Fractions

In [7]:
df_ints.head()

,id_str,hdf5_loc,merging_minor-disturbance_fraction,merging_major-disturbance_fraction,merging_merger_fraction
37,388975_4015,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.051234,0.068933,0.770546
38,388975_4016,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.046352,0.065630,0.794028
175,388982_6338,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.173024,0.325143,0.261890
284,391820_2807,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.139567,0.221040,0.472625
325,388987_6060,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.298189,0.316492,0.140175


In [8]:
df_renorm_ints = (
    df_ints
    .assign(remaining_fraction = df_ints.progress_apply(lambda row: row['merging_minor-disturbance_fraction'] + row['merging_major-disturbance_fraction'] + row['merging_merger_fraction'], axis = 1))
)   

100%|██████████| 157638/157638 [00:03<00:00, 46010.92it/s]


In [9]:
df_renorm_ints.head()

,id_str,hdf5_loc,merging_minor-disturbance_fraction,merging_major-disturbance_fraction,merging_merger_fraction,remaining_fraction
37,388975_4015,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.051234,0.068933,0.770546,0.890714
38,388975_4016,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.046352,0.065630,0.794028,0.906010
175,388982_6338,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.173024,0.325143,0.261890,0.760056
284,391820_2807,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.139567,0.221040,0.472625,0.833232
325,388987_6060,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.298189,0.316492,0.140175,0.754855


In [15]:
df_int_fractions = (
    df_renorm_ints
    .assign(minor_dist = df_renorm_ints.apply(lambda row: (1 / row.remaining_fraction) * row['merging_minor-disturbance_fraction'], axis = 1))
    .assign(major_dist = df_renorm_ints.apply(lambda row: (1 / row.remaining_fraction) * row['merging_major-disturbance_fraction'], axis = 1))
    .assign(merger = df_renorm_ints.apply(lambda row: (1 / row.remaining_fraction) * row['merging_merger_fraction'], axis = 1))
    .drop(columns = ['merging_minor-disturbance_fraction', 'merging_major-disturbance_fraction', 'merging_merger_fraction', 'remaining_fraction'])
)

In [16]:
df_int_fractions

,id_str,hdf5_loc,minor_dist,major_dist,merger
37,388975_4015,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.057520,0.077391,0.865089
38,388975_4016,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.051161,0.072438,0.876401
175,388982_6338,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.227646,0.427788,0.344566
284,391820_2807,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.167501,0.265280,0.567219
325,388987_6060,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.395027,0.419275,0.185698
...,...,...,...,...,...
8689136,444046_118,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.241298,0.374443,0.384259
8689150,441331_4410,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.197326,0.367149,0.435525
8689204,442689_841,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.098107,0.256333,0.645560
8689223,442690_1552,_desi_pytorch_v5_hpv2_train_all_notest_all.hdf5,0.051997,0.114876,0.833126


In [17]:
df_check = (
    df_int_fractions.assign(frac_sum = df_int_fractions.apply(lambda row: row.minor_dist + row.major_dist + row.merger, axis = 1))
)

In [23]:
df_check.query('frac_sum < 0.9999 or frac_sum > 1.00001')

,id_str,hdf5_loc,minor_dist,major_dist,merger,frac_sum


## Selecting Confident Mergers

In [45]:
def clsf_mergers(minor, major, merger):
    if minor > major and minor > merger:
        if minor >= 0.5:
            return 'minor'
        else: 
            return np.nan
    elif major > minor and major > merger:
        if major >= 0.5:
            return 'major'
        else: 
            return np.nan
    elif merger > minor and merger > major:
        if merger >= 0.5:
            return 'merger'
        else: 
            return np.nan
    
    print('Something Wrong!')
    sys.exit()

In [46]:
df_int_clsf = (
    df_int_fractions
    .assign(clsf = df_int_fractions.apply(lambda row: clsf_mergers(row.minor_dist, row.major_dist, row.merger), axis = 1))
)

df_int_selected = df_int_clsf.drop(columns = ['minor_dist', 'major_dist', 'merger'])

In [47]:
df_int_dna = df_int_selected.dropna()

In [48]:
len(df_int_dna)

137963

In [49]:
df_int_dna.clsf.value_counts()

merger    114135
major      23223
minor        605
Name: clsf, dtype: int64

## Saving

In [53]:
df_int_dna.reset_index().drop(columns = ['index']).to_csv(f'C:/Users/oryan/Documents/mergers_in_desi/MiD-revamped/data/selected-interactions.csv')